In this notebook, we want to answer the question: *Is there benefit to combining the models? How should we combine them?*

In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [2]:
import pandas as pd
import numpy as np

from src.paths import DATA_DIR
from src.Qlassifier.results import Results
from src.Qlassifier.prediction import InstructPredictor
from notebooks.modelling import scope

/home/ykip10/projects/Qlassifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pickle

saved_res = Path("saved_results")
# For file loading
model_prefixes = ["instruct", "sent", "tfidf"] 
subject_prefixes = ["chem", "math"]

all_res: list[list[Results, Results]] = []
for mod_pref in model_prefixes:
    res: list[Results, Results] = []
    for subj in subject_prefixes:
        with open(saved_res / f"{mod_pref}_{subj}_res.pkl", "rb") as f:
            res.append(pickle.load(f))
    all_res.append(res)

In [6]:
cols = ["Question", "true_topic_idx"] + [model_pref + "_" + "pred_topic_idx" for model_pref in model_prefixes]

merged_dfs = []
for i in [0, 1]:
    df = pd.DataFrame(
        [all_res[0][i].pred_df["label"], all_res[0][i].pred_df["true_topic_idx"]] +  
        [results[i].pred_df["pred_topic_idx"] for results in all_res],
        index=cols
    ).transpose()
    merged_dfs.append(df)

In [7]:
chem_df = merged_dfs[0]
chem_df.head(5)

,Question,true_topic_idx,instruct_pred_topic_idx,sent_pred_topic_idx,tfidf_pred_topic_idx
0,Question 1,0,0,0,0
1,Question 2,2,2,2,2
2,Question 3,6,6,6,6
3,Question 4,5,2,2,5
4,Question 5,10,3,7,10


For partial weak labelling, which might be what we end up doing here, we want to able to automatically label as many datasets as possible while also keeping the labels high accuracy. For chemistry, we saw that the instructor and sentence transformer model learn pretty much the same patterns, just with the instructor being slightly better. Since this is the case, if we only weakly label when all models agree, many points WON'T be labelled because the sentence transformer "wrong" predictions will erroneously introduce uncertainty, and the point won't be labelled, but the accuracy of the labels won't increase by any amount that's worth it. This can be checked formally.

In [ ]:
def analyse_merged(df, subject_idx: int):
    res = all_res[0][subject_idx]
    # Find the pairwise agreements between model output
    df["tf_instruct"] = (df["instruct_pred_topic_idx"] == df["tfidf_pred_topic_idx"]) 
    df["tf_sentence"] = (df["sent_pred_topic_idx"] == df["tfidf_pred_topic_idx"])
    df["sentence_instruct"] = (df["sent_pred_topic_idx"] == df["instruct_pred_topic_idx"])
    df["all"] = df["tf_instruct"] & df["sentence_instruct"] # All models agree

    df["common_pred"] = df.apply(
        lambda row: row["instruct_pred_topic_idx"] if row["all"] else np.nan, axis=1
    )

    tf_in_df = df[df["tf_instruct"]] 
    tf_sen_df = df[df["tf_sentence"]]
    sen_in_df = df[df["sentence_instruct"]]
    all_df = df[df["all"]]
    comm_dfs =  [tf_in_df, tf_sen_df, sen_in_df, all_df]

    # Concatenate summaries then print
    summaries = []
    for comm_df in comm_dfs: 
        summary = res.summary(by="overall", subset=comm_df.index)
        summary.loc["#Labelled Rows"] = len(comm_df) 
        summaries.append(summary)

    summaries_df = pd.DataFrame(data=summaries, index=["tf_instruct", "tf_sentence", "sentence_instruct", "all"])
    return summaries_df

In [37]:
round(analyse_merged(chem_df, 0), 2)

,Accuracy,Top 3 Accuracy,Macro-Precision,Macro-Recall,Macro-F1-Score,Weighted-Precision,Weighted-Recall,Weighted-F1-Score,#Labelled Rows
tf_instruct,0.88,0.97,0.68,0.67,0.65,0.92,0.88,0.87,32.0
tf_sentence,0.71,0.94,0.56,0.56,0.53,0.74,0.71,0.68,31.0
sentence_instruct,0.55,0.84,0.54,0.45,0.44,0.67,0.55,0.55,49.0
all,0.87,0.96,0.64,0.65,0.63,0.87,0.87,0.85,23.0


As we can see, the sweet spot for accurate labels and number of labelled rows is indeed the model that just uses TF-IDF and INSTRUCTOR agreements. We can, of course, do the same for math.  

In [ ]:
math_df = merged_dfs[1]
round(analyse_merged(math_df, 1), 2)

,Accuracy,Top 3 Accuracy,Macro-Precision,Macro-Recall,Macro-F1-Score,Weighted-Precision,Weighted-Recall,Weighted-F1-Score,#Labelled Rows
tf_instruct,0.86,1.00,0.70,0.69,0.67,0.90,0.86,0.87,28.0
tf_sentence,0.77,1.00,0.66,0.67,0.63,0.85,0.77,0.77,26.0
sentence_instruct,0.82,0.95,0.77,0.81,0.77,0.84,0.82,0.82,39.0
all,0.86,1.00,0.70,0.71,0.68,0.89,0.86,0.86,22.0


This one's a bit of a toss-up between tf_instruct and sentence_instruct; we have to ask ourselves the question, waht do we value more? Accuracy or reduced labour? 